<p estilo="text-align:centro">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Logotipo de Skills Network">
    </a>
</p>


# **Predicción de aterrizaje de la primera etapa del SpaceX Falcon 9**


# Laboratorio práctico: complete el laboratorio API de recopilación de datos


Tiempo estimado necesario: **45** minutos


En este capítulo, predeciremos si la primera etapa del Falcon 9 aterrizará con éxito. SpaceX anuncia en su sitio web lanzamientos de cohetes Falcon 9 con un costo de 62 millones de dólares; Otros proveedores cuestan más de 165 millones de dólares cada uno; gran parte del ahorro se debe a que SpaceX puede reutilizar la primera etapa. Por lo tanto, si podemos determinar si la primera etapa aterrizará, podemos determinar el costo de un lanzamiento. Esta información se puede utilizar si una empresa alternativa quiere ofertar contra SpaceX para el lanzamiento de un cohete. En esta práctica de laboratorio, recopilará y se asegurará de que los datos estén en el formato correcto desde una API. El siguiente es un ejemplo de un lanzamiento exitoso.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


A continuación se muestran varios ejemplos de un aterrizaje fallido:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Se planean la mayoría de los aterrizajes fallidos. Space X realiza un aterrizaje controlado en los océanos.


## Objetivos


En esta práctica de laboratorio, realizará una solicitud de obtención a la API de SpaceX. También realizarás algunas modificaciones y formateo de datos básicos. 

- Solicitud a la API de SpaceX
- Limpiar los datos solicitados.


Instale las siguientes bibliotecas


In [34]:
!pip install requests
!pip install pandas
!pip install numpy


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importar bibliotecas y definir funciones auxiliares


Importaremos las siguientes bibliotecas al laboratorio.


In [35]:
# Solicitudes nos permite realizar solicitudes HTTP que usaremos para obtener datos de una API.
import requests
# Pandas es una biblioteca de software escrita para el lenguaje de programación Python para la manipulación y análisis de datos.
import pandas as pd
# NumPy es una biblioteca para el lenguaje de programación Python, que agrega soporte para matrices y arreglos multidimensionales grandes, junto con una gran colección de funciones matemáticas de alto nivel para operar en estos arreglos.
import numpy as np
# Datetime es una biblioteca que nos permite representar fechas
import datetime

# Configurar esta opción imprimirá todas las columnas de un marco de datos
pd.set_option('display.max_columns', None)
# Configurar esta opción imprimirá todos los datos en una característica
pd.set_option('display.max_colwidth', None)

A continuación definiremos una serie de funciones auxiliares que nos ayudarán a utilizar la API para extraer información utilizando números de identificación en los datos de lanzamiento.

De la columna <code>rocket</code> nos gustaría saber el nombre del booster (refuerzo).


In [36]:
# Toma el conjunto de datos y usa la columna Rocket para llamar a la API y agregar los datos a la lista.
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

Desde la <code>launchpad</code> nos gustaría saber el nombre del sitio de lanzamiento que se está utilizando, la longitud y la latitud.


In [37]:
# Toma el conjunto de datos y usa la columna de la plataforma de lanzamiento para llamar a la API y agregar los datos a la lista.
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
         response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
         Longitude.append(response['longitude'])
         Latitude.append(response['latitude'])
         LaunchSite.append(response['name'])

De la <code>payload</code> nos gustaría conocer la masa de la carga útil y la órbita a la que se dirige.


In [38]:
# Toma el conjunto de datos y usa la columna de cargas útiles para llamar a la API y agregar los datos a las listas.
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

De los <code>Core</code> nos gustaría conocer el resultado del aterrizaje, el tipo de aterrizaje, el número de vuelos con ese núcleo, si se usaron rejillas, si se reutilizó el núcleo, si se usaron patas, la plataforma de aterrizaje utilizada, el bloque del núcleo, que es un número que se usa para separar las versiones de los núcleos, el número de veces que este núcleo específico se ha reutilizado y el número de serie del núcleo.


In [39]:
# Toma el conjunto de datos y usa la columna de núcleos para llamar a la API y agregar los datos a las listas.
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

Ahora comencemos a solicitar datos de lanzamiento de cohetes desde la API de SpaceX con la siguiente URL:


In [40]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [41]:
response = requests.get(spacex_url)

Comprueba el contenido de la respuesta.


In [42]:
#print(response.content)

Debería ver que la respuesta contiene información masiva sobre los lanzamientos de SpaceX. A continuación, intentemos descubrir información más relevante para este proyecto.


### Tarea 1: Solicitar y analizar los datos de lanzamiento de SpaceX mediante la solicitud GET


Para que los resultados JSON solicitados sean más consistentes, usaremos el siguiente objeto de respuesta estática para este proyecto:


In [43]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

Deberíamos ver que la solicitud fue exitosa con el código de respuesta de estado 200.


In [44]:
response.status_code

200

Ahora decodificamos el contenido de la respuesta como Json usando <code>.json()</code> y lo convertimos en un marco de datos de Pandas usando <code>.json_normalize()</code>


In [45]:
data = pd.json_normalize(response.json())

Usando el marco de datos <code>data</code> imprima las primeras 5 filas


In [46]:
data.head(5)

,static_fire_date_utc,static_fire_date_unix,net,window,rocket,success,failures,details,crew,ships,capsules,payloads,launchpad,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,auto_update,tbd,launch_library_id,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,0.0,5e9d0d95eda69955f709d1eb,False,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",True,False,NaN,5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/94/f2/NN6Ph45r_o.png,https://images2.imgbox.com/5b/02/QcxHUb5V_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,NaN,NaN,False,0.0,5e9d0d95eda69955f709d1eb,False,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]","Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",True,False,NaN,5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/f9/4a/ZboXReNb_o.png,https://images2.imgbox.com/80/a2/bkWotCIS_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,NaN,NaN,False,0.0,5e9d0d95eda69955f709d1eb,False,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",True,False,NaN,5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/6c/cb/na1tzhHs_o.png,https://images2.imgbox.com/4a/80/k1oAkY0k_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,0.0,5e9d0d95eda69955f709d1eb,True,[],"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succ

Notarás que muchos de los datos son identificaciones. Por ejemplo, la columna del cohete no tiene información sobre el cohete, solo un número de identificación.

Ahora usaremos la API nuevamente para obtener información sobre los lanzamientos utilizando los ID proporcionados para cada lanzamiento. Específicamente usaremos las columnas <code>rocket</code>, <code>payloads</code>, <code>launchpad</code> y <code>cores</code>.


In [47]:
# Tomemos un subconjunto de nuestro marco de datos manteniendo solo las funciones que queremos y el número de vuelo, y date_utc.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Eliminaremos filas con múltiples núcleos porque son cohetes Falcon con 2 propulsores de cohetes adicionales y filas que tienen múltiples cargas útiles en un solo cohete.
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]

# Dado que las cargas útiles y los núcleos son listas de tamaño 1, también extraeremos el valor único de la lista y reemplazaremos la característica.
data['cores'] = data['cores'].map(lambda x : x[0])
data['payloads'] = data['payloads'].map(lambda x : x[0])

# También queremos convertir date_utc a un tipo de datos de fecha y hora y luego extraer la fecha dejando la hora.
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Utilizando la fecha restringiremos las fechas de los lanzamientos.
data = data[data['date'] <= datetime.date(2020, 11, 13)]

* De <code>Rocket</code> nos gustaría saber el nombre del Booster.

* De la <code>payload</code> nos gustaría saber la masa de la carga útil y la órbita a la que se dirige.

* Desde la <code>launchpad</code> nos gustaría saber el nombre del sitio de lanzamiento que se está utilizando, la longitud y la latitud.

* **De <code>núcleos</code> nos gustaría conocer el resultado del aterrizaje, el tipo de aterrizaje, el número de vuelos con ese núcleo, si se usaron rejillas, si se reutilizó el núcleo, si se usaron patas, la plataforma de aterrizaje utilizada, el bloque del núcleo, que es un número usado para separar las versiones de los núcleos, el número de veces que este núcleo específico se ha reutilizado y el número de serie del núcleo.**

Los datos de estas solicitudes se almacenarán en listas y se utilizarán para crear un nuevo marco de datos.


In [48]:
# variables globales
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

Estas funciones aplicarán las salidas globalmente a las variables anteriores. Echemos un vistazo a la variable <code>BoosterVersion</code>. Antes de aplicar <code>getBoosterVersion</code> la lista está vacía:


In [49]:
BoosterVersion

[]

Ahora, apliquemos el método de función <code> getBoosterVersion</code> para obtener la versión de refuerzo.


In [50]:
# Llame a getBoosterVersion
getBoosterVersion(data)

la lista ahora ha sido actualizada


In [51]:
BoosterVersion[0:5]

['Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 9']

Podemos aplicar el resto de funciones aquí:


In [52]:
# Llame a getLaunchSite
getLaunchSite(data)

In [53]:
# Llame a getPayloadData
getPayloadData(data)

In [54]:
# Llame a getCoreData
getCoreData(data)

Finalmente, construyamos nuestro conjunto de datos utilizando los datos que hemos obtenido. Combinamos las columnas en un diccionario.


In [55]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}


Luego, necesitamos crear un marco de datos de Pandas a partir del diccionario launch_dict.


In [60]:
# Crear datos desde launch_dict
data = pd.DataFrame(launch_dict)

Mostrar el resumen del dataframe


In [61]:
data.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24,Falcon 1,20.0,LEO,Kwajalein Atoll,None None,1,False,False,False,NaN,NaN,0,Merlin1A,167.743129,9.047721
1,2,2007-03-21,Falcon 1,NaN,LEO,Kwajalein Atoll,None None,1,False,False,False,NaN,NaN,0,Merlin2A,167.743129,9.047721
2,4,2008-09-28,Falcon 1,165.0,LEO,Kwajalein Atoll,None None,1,False,False,False,NaN,NaN,0,Merlin2C,167.743129,9.047721
3,5,2009-07-13,Falcon 1,200.0,LEO,Kwajalein Atoll,None None,1,False,False,False,NaN,NaN,0,Merlin3C,167.743129,9.047721
4,6,2010-06-04,Falcon 9,NaN,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857


### Tarea 2: Filtrar el marco de datos para incluir solo los lanzamientos de `Falcon 9`


Finalmente eliminaremos los lanzamientos de Falcon 1 manteniendo solo los lanzamientos de Falcon 9. Filtre el marco de datos usando la columna <code>BoosterVersion</code> para conservar solo los lanzamientos de Falcon 9. Guarde los datos filtrados en un nuevo marco de datos llamado <code>data_falcon9</code>.


In [63]:
data_falcon9 = data[data.BoosterVersion == "Falcon 9"]
data_falcon9

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
4,6,2010-06-04,Falcon 9,NaN,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
5,8,2012-05-22,Falcon 9,525.0,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
6,10,2013-03-01,Falcon 9,677.0,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
7,11,2013-09-29,Falcon 9,500.0,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
8,12,2013-12-03,Falcon 9,3170.0,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,102,2020-09-03,Falcon 9,15600.0,VLEO,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,12,B1060,-80.603956,28.608058
90,103,2020-10-06,Falcon 9,15600.0,VLEO,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,13,B1058,-80.603956,28.608058
91,104,2020-10-18,Falcon 9,15600.0,VLEO,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,12,B1051,-80.603956,28.608058
92,105,2020-10-24,Falcon 9,15600.0,VLEO,CCSFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,5.0,12,B1060,-80.577366,28.561857


Ahora que hemos eliminado algunos valores, debemos restablecer la columna FlgihtNumber.


In [64]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
4,1,2010-06-04,Falcon 9,NaN,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
5,2,2012-05-22,Falcon 9,525.0,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
6,3,2013-03-01,Falcon 9,677.0,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
7,4,2013-09-29,Falcon 9,500.0,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
8,5,2013-12-03,Falcon 9,3170.0,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,86,2020-09-03,Falcon 9,15600.0,VLEO,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,12,B1060,-80.603956,28.608058
90,87,2020-10-06,Falcon 9,15600.0,VLEO,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,13,B1058,-80.603956,28.608058
91,88,2020-10-18,Falcon 9,15600.0,VLEO,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,12,B1051,-80.603956,28.608058
92,89,2020-10-24,Falcon 9,15600.0,VLEO,CCSFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,5.0,12,B1060,-80.577366,28.561857


## Disputa de datos


Podemos ver a continuación que a algunas de las filas les faltan valores en nuestro conjunto de datos.


In [65]:
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        5
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

Antes de que podamos continuar, debemos abordar estos valores faltantes. La columna <code>LandingPad</code> conservará los valores Ninguno para representar cuándo no se utilizaron las plataformas de aterrizaje.


### Tarea 3: Tratar con los valores perdidos


Calcule a continuación la media para <code>PayloadMass</code> usando <code>.mean()</code>. Luego use la media y la función <code>.replace()</code> para reemplazar los valores `np.nan` en los datos con la media que calculó.


In [69]:
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, data_falcon9.PayloadMass.mean())

Debería ver que el número de valores faltantes de <code>PayLoadMass</code> cambia a cero.


Ahora no deberíamos tener valores faltantes en nuestro conjunto de datos excepto en <code>LandingPad</code>.


Ahora podemos exportarlo a un <b>CSV</b> para la siguiente sección, pero para que las respuestas sean consistentes, en la próxima práctica de laboratorio proporcionaremos datos en un rango de fechas preseleccionado.


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


In [70]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)